In [1]:
import re
import pandas as pd
from sklearn.metrics import normalized_mutual_info_score

In [4]:

# Phase 6 (Test-only Evaluation)

INPUT = "phase4_pca_clusters.json"  # Phase 4 output (merged df_all)

# --- Load ---
df = pd.read_json(INPUT, lines=True)

# --- Keep only TEST split ---
df = df[df["dataset"] == "test"].copy()
print("Test rows:", len(df))

# --- Normalize helpers ---
def _canon(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s*>\s*", " > ", s)  # normalize separators
    s = re.sub(r"\s+", " ", s)        # collapse spaces
    return s

# --- Build GT_Path (prefer pathlist_names) ---
for col in ["Level1", "Level2", "Level3", "pathlist_names"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

if "pathlist_names" in df.columns and df["pathlist_names"].str.strip().ne("").any():
    df["GT_Path"] = df["pathlist_names"].map(_canon)
else:
    df["GT_Path"] = (
        df[["Level1", "Level2", "Level3"]]
        .agg(lambda r: " > ".join([p for p in r if p]), axis=1)
        .map(_canon)
    )

# --- Keep non-noise + valid GT ---
mask = (df["HDBSCAN_Cluster"] >= 0) & (df["GT_Path"].str.strip() != "")
df_eval = df[mask].copy()

coverage = len(df_eval) / len(df) if len(df) else 0.0

if df_eval.empty:
    print("No non-noise test points with valid ground-truth path. NMI = NaN")
else:
    # Encode GT strings as categorical codes for NMI
    y_true = df_eval["GT_Path"].astype("category").cat.codes.values
    y_pred = df_eval["HDBSCAN_Cluster"].values

    test_nmi = normalized_mutual_info_score(y_true, y_pred, average_method="arithmetic")
    print(f"Coverage (non-noise/total, test): {coverage:.2%}")
    print(f"Test NMI (vs GT_Path): {test_nmi:.4f}")


Test rows: 153095
Coverage (non-noise/total, test): 58.78%
Test NMI (vs GT_Path): 0.6310


In [6]:

# Phase 6 (Train-only Evaluation)

INPUT = "phase4_pca_clusters.json"  # Phase 4 output (merged df_all)

# --- Load ---
df = pd.read_json(INPUT, lines=True)

# --- Keep only TRAIN split ---
df = df[df["dataset"] == "train"].copy()
print("Train rows:", len(df))

# --- Normalize helpers ---
def _canon(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s*>\s*", " > ", s)  # normalize separators
    s = re.sub(r"\s+", " ", s)        # collapse spaces
    return s

# --- Build GT_Path (prefer pathlist_names) ---
for col in ["Level1", "Level2", "Level3", "pathlist_names"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

if "pathlist_names" in df.columns and df["pathlist_names"].str.strip().ne("").any():
    df["GT_Path"] = df["pathlist_names"].map(_canon)
else:
    df["GT_Path"] = (
        df[["Level1", "Level2", "Level3"]]
        .agg(lambda r: " > ".join([p for p in r if p]), axis=1)
        .map(_canon)
    )

# --- Keep non-noise + valid GT ---
mask = (df["HDBSCAN_Cluster"] >= 0) & (df["GT_Path"].str.strip() != "")
df_eval = df[mask].copy()

coverage = len(df_eval) / len(df) if len(df) else 0.0

if df_eval.empty:
    print("No non-noise test points with valid ground-truth path. NMI = NaN")
else:
    # Encode GT strings as categorical codes for NMI
    y_true = df_eval["GT_Path"].astype("category").cat.codes.values
    y_pred = df_eval["HDBSCAN_Cluster"].values

    test_nmi = normalized_mutual_info_score(y_true, y_pred, average_method="arithmetic")
    print(f"Coverage (non-noise/total, train): {coverage:.2%}")
    print(f"Train NMI (vs GT_Path): {test_nmi:.4f}")


Train rows: 489902
Coverage (non-noise/total, train): 59.14%
Train NMI (vs GT_Path): 0.6300


In [3]:
pip install plotly

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 72.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.1/392.1 kB 73.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Interactive Visualization of the Hierarchy tree by unsupervised product classifiaction

In [2]:
# === Phase 7: Interactive Top-Down TREE (Plotly, Jupyter-Friendly) ===
# pip install plotly networkx

import json
from collections import defaultdict
from pathlib import Path

import networkx as nx
from networkx.readwrite import json_graph
import plotly.graph_objects as go
import plotly.io as pio

# ---------------------------
# Auto-detect environment and set renderer
# ---------------------------
def _ensure_plotly_renderer():
    try:
        import IPython
        ip = IPython.get_ipython()
        if ip and "ipykernel" in str(type(ip)):
            pio.renderers.default = "notebook_connected"  # fully interactive inline
        else:
            pio.renderers.default = "browser"  # fallback to external browser tab
    except Exception:
        pio.renderers.default = "browser"

_ensure_plotly_renderer()

# ---------------------------
# Inputs / toggles
# ---------------------------
GRAPH_JSON_IN = "phase5_taxonomy_graph.json"   # from Phase 5
FOCUS_L1 = None        # e.g. "Computers & Electronics" or None for ALL
INCLUDE_LEVEL4 = True  # include product leaves
SHOW_LABELS = True     # draw text labels on nodes (except leaves)
MAX_LABEL = 40         # truncate labels for readability
HTML_OUT = Path(GRAPH_JSON_IN).with_name("phase7_tree_interactive.html")

# ---------------------------
# 1) Load graph
# ---------------------------
with open(GRAPH_JSON_IN, "r", encoding="utf-8") as f:
    data = json.load(f)
G = json_graph.node_link_graph(data, directed=True, multigraph=False)

print("Loaded:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

# ---------------------------
# 2) Optional filtering
# ---------------------------
if FOCUS_L1 is not None:
    roots = [n for n, d in G.nodes(data=True) if d.get("type") == "level1" and d.get("label") == FOCUS_L1]
    if not roots:
        raise ValueError(f"Level1 '{FOCUS_L1}' not found.")
    keep = set([roots[0]])
    keep.update(nx.descendants(G, roots[0]))
    G = G.subgraph(keep).copy()
    print(f"Focused on L1='{FOCUS_L1}':", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

if not INCLUDE_LEVEL4:
    to_drop = [n for n, d in G.nodes(data=True) if d.get("type") == "leaf"]
    G.remove_nodes_from(to_drop)
    print("Removed Level4 leaves:", len(to_drop))

# ---------------------------
# 3) Build a tree-like layout
# ---------------------------
level_order = {"level1": 1, "level2": 2, "level3": 3, "cluster": 4, "leaf": 5}
def node_level(n):
    return level_order.get(G.nodes[n].get("type", "cluster"), 4)

parents = {n: [] for n in G.nodes()}
for u, v in G.edges():
    parents[v].append(u)

primary_parent = {}
for n in G.nodes():
    ps = parents[n]
    if not ps:
        primary_parent[n] = None
    else:
        ps.sort(key=lambda p: (node_level(p), G.nodes[p].get("label","")))
        primary_parent[n] = ps[0]

children = defaultdict(list)
for n, p in primary_parent.items():
    if p is not None:
        children[p].append(n)

roots = [n for n in G.nodes() if primary_parent[n] is None and G.nodes[n].get("type") == "level1"]
if not roots:
    roots = [n for n in G.nodes() if primary_parent[n] is None]

for n in children:
    children[n].sort(key=lambda x: G.nodes[x].get("label",""))
roots.sort(key=lambda x: G.nodes[x].get("label",""))

def subtree_width(n):
    if n not in children or len(children[n]) == 0:
        return 1
    return sum(subtree_width(c) for c in children[n])

xpos, ypos = {}, {}
Y_SPACING = 1.0

def place_subtree(n, x_left):
    w = subtree_width(n)
    x_center = x_left + w / 2.0
    xpos[n] = x_center
    ypos[n] = -node_level(n) * Y_SPACING
    cur = x_left
    for c in children.get(n, []):
        wc = subtree_width(c)
        place_subtree(c, cur)
        cur += wc

x_cursor = 0.0
GAP_BETWEEN_ROOTS = 2.0
for r in roots:
    w = subtree_width(r)
    place_subtree(r, x_cursor)
    x_cursor += w + GAP_BETWEEN_ROOTS

for n in G.nodes():
    if n not in xpos:
        xpos[n] = x_cursor
        ypos[n] = -node_level(n) * Y_SPACING
        x_cursor += 1.0

# ---------------------------
# 4) Styling / hovers
# ---------------------------
def node_color(d):
    t = d.get("type")
    return {
        "level1": "#7ec8e3",
        "level2": "#a0e493",
        "level3": "#f6b26b",
        "cluster": "#b3b3b3",
        "leaf": "#f4b6c2",
    }.get(t, "#cccccc")

def node_size(d):
    base = 18
    if d.get("type") == "cluster":
        return min(40, base + 0.03 * d.get("size", 1))
    if d.get("type") == "leaf":
        return 16
    return 18

def cluster_label_text(raw):
    return raw.split(":", 1)[-1].strip() if ":" in raw else raw

def trunc(s, n=MAX_LABEL):
    s = s or ""
    return (s[:n-1] + "…") if len(s) > n else s

def hover_text(n, d):
    t = d.get("type", "")
    raw_label = d.get("label", "") or n
    tfidf_label = cluster_label_text(raw_label) if t == "cluster" else raw_label
    top_terms = d.get("top_terms", [])
    keywords = d.get("keywords", [])
    lines = [f"<b>{tfidf_label}</b>", f"Type: {t}"]
    if t == "cluster":
        lines.append(f"Cluster ID: {d.get('cluster_id','')} | Size: {d.get('size','')}")
    if top_terms:
        lines.append("Top terms: " + ", ".join(top_terms[:8]))
    if keywords:
        lines.append("Keywords: " + ", ".join(keywords[:8]))
    return "<br>".join(lines)

# ---------------------------
# 5) Build Plotly traces
# ---------------------------
edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = xpos[u], ypos[u]
    x1, y1 = xpos[v], ypos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode="lines",
    line=dict(width=1, color="rgba(120,120,120,0.35)"),
    hoverinfo="none",
    showlegend=False,
)

xs, ys, sizes, colors, hovers = [], [], [], [], []
for n, d in G.nodes(data=True):
    xs.append(xpos[n])
    ys.append(ypos[n])
    sizes.append(node_size(d))
    colors.append(node_color(d))
    hovers.append(hover_text(n, d))

node_trace = go.Scatter(
    x=xs, y=ys,
    mode="markers",
    marker=dict(size=sizes, color=colors, line=dict(width=1, color="rgba(60,60,60,0.8)")),
    text=hovers,
    hovertemplate="%{text}<extra></extra>",
    showlegend=False,
)

label_traces = []
if SHOW_LABELS:
    label_x, label_y, label_txt = [], [], []
    for n, d in G.nodes(data=True):
        if d.get("type") == "leaf":
            continue
        raw = d.get("label", "") or str(n)
        txt = cluster_label_text(raw) if d.get("type") == "cluster" else raw
        label_x.append(xpos[n])
        label_y.append(ypos[n] + 0.15)
        label_txt.append(trunc(txt, MAX_LABEL))
    label_traces.append(go.Scatter(
        x=label_x, y=label_y,
        mode="text",
        text=label_txt,
        textposition="top center",
        textfont=dict(size=10),
        hoverinfo="none",
        showlegend=False,
    ))

# ---------------------------
# 6) Render & Save
# ---------------------------
fig = go.Figure(data=[edge_trace, node_trace] + label_traces)
fig.update_layout(
    title="Phase 7 — Top-Down Taxonomy Tree (Interactive)",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20),
    width=1400, height=900,
)

fig.write_html(str(HTML_OUT), include_plotlyjs="cdn", full_html=True)
print(f"Saved interactive HTML to: {HTML_OUT.resolve()}")

fig.show(config={"responsive": True})


/usr/local/lib/python3.11/dist-packages/networkx/readwrite/json_graph/node_link.py:287: FutureWarning:


The default value will be changed to `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_graph(data, edges="links") to preserve current behavior, or
  nx.node_link_graph(data, edges="edges") for forward compatibility.



Loaded: 9727 nodes, 10201 edges
Saved interactive HTML to: /home/jovyan/phase7_tree_interactive.html
